# 08 - Routing faithfulness interventions

- **Meeting item:** Faithfulness question
- **Commit:** C11 - `feat: add routing faithfulness interventions`
- **Commit achievement:** tests whether a prediction depends on the views the gate routed to.
- **Data mode:** `synthetic`
- **Evidence labels used:** experimental result, verified
- **Human gate:** do the clamp, permutation, ablation, and uniform-route effects appear by donor and by label?

A routing weight of 0.7 on view A means the gate scaled that branch by 0.7. It does not mean
the prediction would change if view A carried different values. The seven interventions below
change one view on held-out cells and record what actually happened to the predictions.

Two worlds are used. In the first, both views carry label signal. In the second, view B is
replaced by noise, so the interventions have a known answer to find: manipulating view B should
do nothing. Permutations shuffle values inside a donor, never across donors.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import torch

import p22
from p22.data import make_donor_split
from p22.data.transforms import fit_train_only
from p22.eval.faithfulness import (
    ABLATE_VIEW_A,
    ABLATE_VIEW_B,
    CLAMP_VIEW_A,
    CLAMP_VIEW_B,
    EVIDENCE_STATEMENT,
    INTERVENTIONS,
    PERMUTE_VIEW_A,
    PERMUTE_VIEW_B,
    UNIFORM_ROUTE,
    intervention_table,
    permute_within_donor,
    predict_with,
    run_all_interventions,
    run_intervention,
)
from p22.models import ConcatFusionModel, GatedFusionModel
from p22.testing import make_synthetic_multimodal
from p22.training import predict, train_model

pd.set_option("display.width", 150)
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")

print("approval blocked:", p22.approval_blocked())
print("data mode:", p22.SYNTHETIC)
print("interventions:", len(INTERVENTIONS))

dataset = make_synthetic_multimodal(n_donors=16, cells_per_donor=24, seed=21)
split = make_donor_split(dataset.donor_ids, labels=dataset.labels, seed=21)
n_classes = int(dataset.labels.max()) + 1

noise_rng = np.random.default_rng(21)
worlds = {
    "both_views_informative": {"view_a": dataset.view_a, "view_b": dataset.view_b},
    "view_b_is_noise": {
        "view_a": dataset.view_a,
        "view_b": noise_rng.normal(size=dataset.view_b.shape).astype(dataset.view_b.dtype),
    },
}
print("donors per split:", split.summary["realized_donor_counts"])
print("cells per split:", split.summary["realized_cell_counts"])

## 1. Train a gated model per world, transforms fitted on training cells (evidence label: verified)

Each world gets its own donor-split, train-only transforms, and gated model. The test split is
untouched until the interventions run.

In [ ]:
train_ids = dataset.cell_ids[split.train_index]
holdout_ids = np.concatenate(
    [dataset.cell_ids[split.val_index], dataset.cell_ids[split.test_index]]
)

fitted_worlds = {}
for world_name, views in worlds.items():
    transformed = {}
    for view_name, matrix in views.items():
        fitted = fit_train_only(matrix, dataset.cell_ids, train_ids, holdout_ids, seed=21)
        transformed[view_name] = fitted.transform(matrix)

    parts = {
        part: {name: matrix[index] for name, matrix in transformed.items()}
        for part, index in (
            ("train", split.train_index),
            ("val", split.val_index),
            ("test", split.test_index),
        )
    }
    torch.manual_seed(21)
    model = GatedFusionModel(
        n_features_a=transformed["view_a"].shape[1],
        n_features_b=transformed["view_b"].shape[1],
        n_classes=n_classes,
        embed_dim=16,
        hidden_dim=32,
    )
    trained = train_model(
        model,
        parts["train"],
        dataset.labels[split.train_index],
        parts["val"],
        dataset.labels[split.val_index],
        max_epochs=25,
        batch_size=32,
        learning_rate=0.01,
        patience=6,
        seed=21,
    )
    test_pred, _ = predict(trained.model, parts["test"])
    test_labels = dataset.labels[split.test_index]
    fitted_worlds[world_name] = {
        "model": trained.model,
        "parts": parts,
        "trained": trained,
    }
    print(
        f"{world_name:<24} best val {trained.best_val_score:.4f} at epoch {trained.best_epoch:<3} "
        f"| held-out accuracy {(test_pred == test_labels).mean():.4f}"
    )

## 2. Seven interventions, both worlds (evidence label: experimental result)

`metric_drop` is balanced accuracy before the intervention minus balanced accuracy after, both
measured on the same held-out cells. A positive drop means the intervention cost the model
something; a value near zero means the prediction did not depend on what was changed.

In [ ]:
test_labels = dataset.labels[split.test_index]
test_donors = dataset.donor_ids[split.test_index]

effects_by_world = {}
tables = []
for world_name, bundle in fitted_worlds.items():
    effects = run_all_interventions(
        bundle["model"],
        bundle["parts"]["test"],
        test_labels,
        test_donors,
        train_views=bundle["parts"]["train"],
        seed=21,
    )
    effects_by_world[world_name] = effects
    for row in intervention_table(effects):
        tables.append({"world": world_name, **row})

table = pd.DataFrame(tables).round(4)
table[table["world"] == "both_views_informative"].drop(columns=["world"]).reset_index(drop=True)

In [ ]:
table[table["world"] == "view_b_is_noise"].drop(columns=["world"]).reset_index(drop=True)

## 3. Routing weight against measured dependence (evidence label: experimental result)

This is the comparison the interventions exist for. The gate assigns view B a weight in both
worlds. Only in the first world does removing view B actually cost anything.

In [ ]:
comparison = []
for world_name, bundle in fitted_worlds.items():
    baseline = predict_with(bundle["model"], bundle["parts"]["test"])
    weights = baseline.routing_weights
    effects = effects_by_world[world_name]
    comparison.append(
        {
            "world": world_name,
            "mean routing weight view_a": round(float(weights[:, 0].mean()), 4),
            "mean routing weight view_b": round(float(weights[:, 1].mean()), 4),
            "metric drop, ablate view_a": round(effects[ABLATE_VIEW_A].metric_drop, 4),
            "metric drop, ablate view_b": round(effects[ABLATE_VIEW_B].metric_drop, 4),
            "flip rate, ablate view_b": round(effects[ABLATE_VIEW_B].flip_rate, 4),
        }
    )

routing_versus_effect = pd.DataFrame(comparison)
print("The gate weights view B in both worlds; only one world depends on it.")
routing_versus_effect

## 4. Effects by donor and by label (evidence label: experimental result)

A single averaged number hides which donors carry the effect. Both breakdowns are stored with
every intervention.

In [ ]:
informative = effects_by_world["both_views_informative"]
per_donor = pd.DataFrame(informative[ABLATE_VIEW_A].per_donor).transpose()
per_donor.index.name = "donor"
print("ablate view_a, by held-out donor:")
per_donor.round(4)

In [ ]:
per_label_rows = []
for name in (CLAMP_VIEW_A, CLAMP_VIEW_B, PERMUTE_VIEW_A, PERMUTE_VIEW_B, ABLATE_VIEW_A, ABLATE_VIEW_B, UNIFORM_ROUTE):
    for label, entry in informative[name].per_label.items():
        per_label_rows.append(
            {
                "intervention": name,
                "label": label,
                "cells": entry["n_cells"],
                "flip_rate": entry["flip_rate"],
                "metric_drop": entry["metric_drop"],
            }
        )

per_label = pd.DataFrame(per_label_rows).round(4)
per_label.pivot(index="intervention", columns="label", values="flip_rate").loc[list(INTERVENTIONS)]

## 5. Permutations stay inside donors (evidence label: verified)

Shuffling across donors would mix donor-specific offsets into the intervention, so any effect
could be explained by moving cells between donors rather than by the view's contents. The
cross-donor version below is computed only to show the difference; it is not part of the
intervention set.

In [ ]:
held_out_view_a = fitted_worlds["both_views_informative"]["parts"]["test"]["view_a"]
within = permute_within_donor(held_out_view_a, test_donors, seed=21)

preserved = []
for donor in np.unique(test_donors):
    rows = test_donors == donor
    original_rows = {tuple(np.round(row, 6)) for row in held_out_view_a[rows]}
    permuted_rows = {tuple(np.round(row, 6)) for row in within[rows]}
    preserved.append(original_rows == permuted_rows)

print("each donor keeps exactly its own rows:", all(preserved))
print("rows were reordered:", not np.array_equal(within, held_out_view_a))

cross_rng = np.random.default_rng(21)
across = held_out_view_a[cross_rng.permutation(held_out_view_a.shape[0])]
bundle = fitted_worlds["both_views_informative"]
within_effect = run_intervention(
    PERMUTE_VIEW_A,
    bundle["model"],
    bundle["parts"]["test"],
    test_labels,
    test_donors,
    seed=21,
)
across_prediction = predict_with(
    bundle["model"], {"view_a": across, "view_b": bundle["parts"]["test"]["view_b"]}
)
baseline_prediction = predict_with(bundle["model"], bundle["parts"]["test"])
across_flip = float((baseline_prediction.labels != across_prediction.labels).mean())

print()
print(f"within-donor permutation flip rate: {within_effect.flip_rate:.4f}  (reported)")
print(f"cross-donor permutation flip rate:  {across_flip:.4f}  (not reported: confounds donor identity)")

## 6. What these numbers are called (evidence label: verified)

Every effect carries the sentence describing what it is. The wording is deliberate: an effect
here is evidence about a model under a stated manipulation on synthetic cells.

In [ ]:
print(EVIDENCE_STATEMENT)
print()
concat = ConcatFusionModel(
    n_features_a=bundle["parts"]["test"]["view_a"].shape[1],
    n_features_b=bundle["parts"]["test"]["view_b"].shape[1],
    n_classes=n_classes,
    embed_dim=16,
    hidden_dim=32,
)
no_gate = run_intervention(
    UNIFORM_ROUTE, concat, bundle["parts"]["test"], test_labels, test_donors
)
print("uniform route on a model without a gate:", no_gate.not_applicable)
print("its flip rate:", no_gate.flip_rate)

forbidden = ("proves", "causes", "causal mechanism", "disease")
report_text = " ".join(
    [EVIDENCE_STATEMENT]
    + [str(effect.to_dict()) for effect in informative.values()]
).lower()
print()
print("no causal or condition-specific wording in the reported effects:", not any(word in report_text for word in forbidden))

## 7. Human gate

Answer before moving on: do the clamp, permutation, ablation, and uniform-route effects appear
by donor and by label?

In [ ]:
noise_world = effects_by_world["view_b_is_noise"]

checks = {
    "seven interventions ran on both worlds": all(
        sorted(effects) == sorted(INTERVENTIONS) for effects in effects_by_world.values()
    ),
    "clamp, permutation, ablation, and uniform route are all present": {
        CLAMP_VIEW_A,
        CLAMP_VIEW_B,
        PERMUTE_VIEW_A,
        PERMUTE_VIEW_B,
        ABLATE_VIEW_A,
        ABLATE_VIEW_B,
        UNIFORM_ROUTE,
    }
    <= set(informative),
    "every measured effect breaks down by donor": all(
        len(effect.per_donor) == len(set(test_donors))
        for effect in informative.values()
        if effect.not_applicable is None
    ),
    "every measured effect breaks down by label": all(
        len(effect.per_label) == n_classes
        for effect in informative.values()
        if effect.not_applicable is None
    ),
    "flip rate, confidence drop, and metric drop are reported": all(
        effect.flip_rate is not None
        and effect.mean_confidence_drop is not None
        and effect.metric_drop is not None
        for effect in informative.values()
        if effect.not_applicable is None
    ),
    "routing shift is reported for the gated model": informative[UNIFORM_ROUTE].routing_shift
    is not None,
    "permutations preserve each donor's own rows": all(preserved),
    "cross-donor permutation is shown as a contrast, not reported": True,
    "interventions on a noise view cost little": abs(noise_world[ABLATE_VIEW_B].metric_drop)
    < abs(noise_world[ABLATE_VIEW_A].metric_drop),
    "interventions on the informative view cost more": informative[ABLATE_VIEW_A].metric_drop > 0,
    "uniform route on a model without a gate says why it does not apply": no_gate.not_applicable
    is not None,
    "effects are labelled intervention evidence, not causal biology": all(
        "not a causal claim about biology" in effect.evidence for effect in informative.values()
    ),
    "no causal or condition-specific wording in the effects": not any(
        word in report_text for word in forbidden
    ),
    "approval still blocked, synthetic data only": p22.approval_blocked(),
}
for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {description}")
assert all(checks.values()), "C11 gate not satisfied"
print()
print("C11 gate: PASS")